# Persian ASR seven-model demo

This notebook draws the configured number of reproducible random rows from each dataset's **`test.tsv`** and runs the exact same clips through seven ASR systems: normal Whisper-small, multiconditioned Whisper-small, fusion, Whisper-medium, normal FastConformer, multiconditioned FastConformer, and multiconditioned Whisper-small additionally trained with IranSeda.

Run cells from top to bottom. Each model cell loads only its own checkpoint and releases it, runs garbage collection, and empties the CUDA cache before returning. Predictions remain in CPU/Python memory for the final comparison.

In [ ]:
from __future__ import annotations

import gc
import os
import random
import sys
import time
from pathlib import Path
from typing import Any, Iterable, Sequence

import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display

pd.set_option('display.max_colwidth', None)


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'ml').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the MS-Thesis repository.')


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if not torch.cuda.is_available():
    raise RuntimeError('This demo is CUDA-only, but torch.cuda.is_available() is false.')
DEVICE = 'cuda'
print(f'Repository: {REPO_ROOT}')
print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## Configuration

Edit dataset display names, dataset directories, sample count, and checkpoint paths here. Dataset paths may be absolute or relative to the repository. A dataset must follow the repository contract: `test.tsv` with `path` and `sentence` columns plus a `clips/` directory.

In [ ]:
RANDOM_SEED = 1337
SAMPLES_PER_DATASET = 2
BATCH_SIZE = 1                 # Reduce to 1 if a long clip approaches GPU limits.
SAMPLE_RATE = 16_000
MAX_NEW_TOKENS = 225
FASTCONFORMER_MAX_BATCH_SECONDS = 30.0
SHOW_AUDIO_PLAYERS = True

# Each name is only a display label; each path identifies a dataset directory.
TEST_DATASETS = [
    {'name': 'AGFarsdat', 'path': 'data/AGFarsdat_test_normalized'},
    {'name': 'Common Voice 25', 'path': 'data/cv-corpus-25.0'},
    {'name': 'FLEURS', 'path': 'data/fleurs-normalized'},
    {'name': 'PersianSpeech', 'path': 'data/PersianSpeech_test'},
    {'name': 'Persian Speech Corpus', 'path': 'data/persian-speech-corpus-test'},
]

# Whisper-small models
WHISPER_NORMAL_CHECKPOINT = 'models/asr/whisper-small-normal/runs/whisper-small-fa/best'
WHISPER_NORMAL_PROCESSOR = 'openai/whisper-small'
WHISPER_MULTICONDITIONED_CHECKPOINT = 'models/asr/whisper-small-deg-v2/runs/whisper-small-fa/best'
WHISPER_MULTICONDITIONED_PROCESSOR = 'openai/whisper-small'
WHISPER_MULTICONDITIONED_IRANSEDA_CHECKPOINT = 'models/asr/whisper-small-iranseda/runs/whisper-small-fa/best'
WHISPER_MULTICONDITIONED_IRANSEDA_PROCESSOR = 'openai/whisper-small'

# Dual-view fusion (the Stage-2 checkpoint includes its jointly trained backbone).
FUSION_CHECKPOINT = 'models/asr/fusion/run_004/checkpoints/stage2_joint/best.pt'
FUSION_BASE_ASR = 'openai/whisper-small'
FUSION_MODEL_NAME = 'openai/whisper-small'
FUSION_PROCESSOR = 'openai/whisper-small'
FUSION_MIXED_PRECISION = False  # run_004 was trained in FP32; avoids unstable custom-front-end AMP.
FUSION_VIEW_MODE = 'fusion'      # fusion | noisy | enhanced
FUSION_GATE_OVERRIDE = None      # None or a float in [0, 1], only in fusion mode

# Whisper-medium
WHISPER_MEDIUM_CHECKPOINT = 'models/asr/whisper-medium/runs/whisper-medium-fa/best'
WHISPER_MEDIUM_PROCESSOR = 'openai/whisper-medium'

# Standalone FastConformer .pt bundles (a .nemo archive also works).
FASTCONFORMER_NORMAL_CHECKPOINT = 'models/asr/fastconformer/runs/fastconformer-fa/best.pt'
FASTCONFORMER_MULTICONDITIONED_CHECKPOINT = 'models/asr/fastconformer_deg_v2/runs/fastconformer-fa/best.pt'

WHISPER_LANGUAGE = 'Persian'
WHISPER_TASK = 'transcribe'

In [ ]:
from ml.asr.train_whisper_small import (
    WhisperExample, character_error_rate, load_split_examples, word_error_rate,
)


def repo_path(value: str | Path) -> Path:
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (REPO_ROOT / path).resolve()


def local_checkpoint(value: str | Path, label: str) -> Path:
    path = repo_path(value)
    if not path.exists():
        raise FileNotFoundError(f'{label} not found: {path}')
    return path


def model_source(value: str | Path) -> str:
    # Resolve a local checkpoint/processor; leave a Hugging Face Hub ID unchanged.
    expanded = Path(value).expanduser()
    for candidate in (expanded, REPO_ROOT / expanded):
        if candidate.exists():
            return str(candidate.resolve())
    text = str(value)
    if expanded.is_absolute() or text.startswith(('./', '../', 'models/', 'artifacts/')):
        raise FileNotFoundError(f'Local model source not found: {repo_path(value)}')
    return text


def batched(items: Sequence[Any], size: int) -> Iterable[Sequence[Any]]:
    if size < 1:
        raise ValueError('BATCH_SIZE must be at least 1.')
    for start in range(0, len(items), size):
        yield items[start:start + size]


def sample_test_rows(dataset_specs: Sequence[dict[str, str]], count: int, seed: int):
    if count < 1:
        raise ValueError('SAMPLES_PER_DATASET must be at least 1.')
    selected: list[WhisperExample] = []
    names: list[str] = []
    for dataset_index, spec in enumerate(dataset_specs):
        dataset_dir = repo_path(spec['path'])
        # This project loader validates test.tsv, its path/sentence columns, clips/,
        # and every referenced audio path. No train/dev row can enter the sample.
        candidates = load_split_examples([dataset_dir], 'test')
        if len(candidates) < count:
            raise ValueError(f"{spec['name']} has {len(candidates)} usable test rows; requested {count}.")
        chosen = random.Random(seed + dataset_index).sample(candidates, count)
        selected.extend(chosen)
        names.extend([spec['name']] * len(chosen))
    return selected, names


def synchronize_cuda() -> None:
    torch.cuda.synchronize()


def empty_cuda_cache() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    print(
        'CUDA after cleanup: '
        f'{torch.cuda.memory_allocated() / 2**20:.1f} MiB allocated, '
        f'{torch.cuda.memory_reserved() / 2**20:.1f} MiB reserved'
    )


RESULTS: dict[str, dict[str, Any]] = {}


def save_and_show_results(
    model_name: str, hypotheses: Sequence[str], elapsed: float, details: Sequence[str] | None = None
) -> pd.DataFrame:
    if len(hypotheses) != len(selected_examples):
        raise RuntimeError(f'{model_name} returned {len(hypotheses)} predictions for {len(selected_examples)} clips.')
    references = [example.transcript for example in selected_examples]
    rows = []
    details = list(details or [''] * len(hypotheses))
    for dataset, example, reference, hypothesis, detail in zip(
        selected_dataset_names, selected_examples, references, hypotheses, details, strict=True
    ):
        rows.append({
            'dataset': dataset, 'file': example.audio_path.name, 'reference': reference,
            'hypothesis': hypothesis, 'detail': detail,
        })
    frame = pd.DataFrame(rows)
    RESULTS[model_name] = {
        'hypotheses': list(hypotheses), 'elapsed_seconds': elapsed,
        'wer': word_error_rate(references, list(hypotheses)),
        'cer': character_error_rate(references, list(hypotheses)),
    }
    display(Markdown(
        f"**{model_name}** — {elapsed:.2f} s total, "
        f"WER `{RESULTS[model_name]['wer']:.4f}`, CER `{RESULTS[model_name]['cer']:.4f}`"
    ))
    display(
        frame.style.set_properties(
            subset=['reference', 'hypothesis'],
            **{'white-space': 'pre-wrap', 'text-align': 'right', 'min-width': '24rem'},
        )
    )
    return frame


def run_whisper(model_label: str, checkpoint: str | Path, processor_name: str | Path) -> pd.DataFrame:
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    from ml.asr.whisper_features import waveform_to_log_mel
    from ml.fusion.train_fusion import configure_whisper_generation
    from ml.utils.audio import load_audio, resample_audio, to_mono

    processor = None
    model = None
    try:
        processor_source = model_source(processor_name)
        processor = WhisperProcessor.from_pretrained(
            processor_source, language=WHISPER_LANGUAGE, task=WHISPER_TASK
        )
        model = WhisperForConditionalGeneration.from_pretrained(model_source(checkpoint))
        model = model.to(device=DEVICE, dtype=torch.float16).eval()
        configure_whisper_generation(model, WHISPER_LANGUAGE, WHISPER_TASK)
        hypotheses: list[str] = []
        synchronize_cuda()
        started = time.perf_counter()
        for batch in batched(selected_examples, BATCH_SIZE):
            features = []
            for example in batch:
                waveform, source_rate = load_audio(example.audio_path)
                waveform = to_mono(waveform)
                if int(source_rate) != SAMPLE_RATE:
                    waveform = resample_audio(waveform, int(source_rate), SAMPLE_RATE)
                features.append(waveform_to_log_mel(
                    waveform, sample_rate=SAMPLE_RATE, model_name=processor_source
                ))
            input_features = torch.stack(features).to(DEVICE, dtype=torch.float16)
            with torch.inference_mode(), torch.amp.autocast('cuda', dtype=torch.float16):
                token_ids = model.generate(input_features=input_features, max_new_tokens=MAX_NEW_TOKENS)
            hypotheses.extend(processor.batch_decode(token_ids.cpu(), skip_special_tokens=True))
            del input_features, token_ids, features
        synchronize_cuda()
        elapsed = time.perf_counter() - started
        return save_and_show_results(model_label, hypotheses, elapsed)
    finally:
        if model is not None:
            model.to('cpu')
        del model, processor
        empty_cuda_cache()


def run_fusion() -> pd.DataFrame:
    # This follows ml.fusion.eval_fusion.run_evaluation's inference path.
    from transformers import WhisperTokenizer
    from ml.fusion.eval_fusion import (
        VIEW_MODES, configure_generation, load_fusion_model, resolve_existing_path,
        resolve_source, transcribe_examples, use_amp,
    )
    from ml.fusion.model import is_loadable_checkpoint

    tokenizer = None
    model = None
    try:
        checkpoint = resolve_existing_path(FUSION_CHECKPOINT)
        base_asr_checkpoint = resolve_source(FUSION_BASE_ASR)
        if not is_loadable_checkpoint(base_asr_checkpoint):
            raise FileNotFoundError(
                f'Fusion base ASR checkpoint is not loadable: {base_asr_checkpoint!r}'
            )
        model_name = str(FUSION_MODEL_NAME)
        processor_name = resolve_source(FUSION_PROCESSOR or model_name)
        view_mode = str(FUSION_VIEW_MODE)
        if view_mode not in VIEW_MODES:
            raise ValueError(f'FUSION_VIEW_MODE must be one of {VIEW_MODES}, got {view_mode!r}')
        gate_override = (
            None if FUSION_GATE_OVERRIDE is None else float(FUSION_GATE_OVERRIDE)
        )
        if gate_override is not None and (view_mode != 'fusion' or not 0.0 <= gate_override <= 1.0):
            raise ValueError('FUSION_GATE_OVERRIDE must be in [0, 1] and requires fusion view mode.')
        amp_enabled = use_amp(FUSION_MIXED_PRECISION, DEVICE)

        tokenizer = WhisperTokenizer.from_pretrained(processor_name)
        model, backbone_included = load_fusion_model(
            checkpoint, base_asr_checkpoint=base_asr_checkpoint, model_name=model_name
        )
        if not backbone_included:
            print(
                'Warning: checkpoint has no Whisper backbone; using '
                f'{base_asr_checkpoint}. This must match the backbone used for fusion training.'
            )
        configure_generation(model, WHISPER_LANGUAGE, WHISPER_TASK)
        print(
            f'Fusion checkpoint={checkpoint} | backbone_included={backbone_included} | '
            f'processor={processor_name} | AMP={amp_enabled} | view={view_mode}'
        )

        synchronize_cuda()
        started = time.perf_counter()
        hypotheses, enhanced_weights = transcribe_examples(
            model, selected_examples, tokenizer, device=DEVICE, sample_rate=SAMPLE_RATE,
            model_name=model_name, batch_size=BATCH_SIZE,
            generation_max_length=MAX_NEW_TOKENS, amp_enabled=amp_enabled,
            view_mode=view_mode, gate_override=gate_override,
        )
        synchronize_cuda()
        elapsed = time.perf_counter() - started
        details = [f'enhanced={weight:.4f}; noisy={1.0 - weight:.4f}' for weight in enhanced_weights]
        if len(details) != len(hypotheses):
            details = [
                f'view={view_mode}; backbone_in_checkpoint={backbone_included}'
            ] * len(hypotheses)
        return save_and_show_results('3. Fusion model', hypotheses, elapsed, details)
    finally:
        if model is not None:
            model.to('cpu')
        del model, tokenizer
        empty_cuda_cache()


def run_fastconformer(model_label: str, checkpoint: str | Path) -> pd.DataFrame:
    from ml.asr.eval_fastconformer import load_fastconformer

    model = None
    try:
        model = load_fastconformer(local_checkpoint(checkpoint, model_label), DEVICE).eval()
        synchronize_cuda()
        started = time.perf_counter()
        hypotheses = model.transcribe(
            [str(example.audio_path) for example in selected_examples],
            batch_size=BATCH_SIZE, device=DEVICE, target_sr=SAMPLE_RATE, progress=True,
            max_batch_seconds=FASTCONFORMER_MAX_BATCH_SECONDS,
        )
        synchronize_cuda()
        elapsed = time.perf_counter() - started
        return save_and_show_results(model_label, hypotheses, elapsed)
    finally:
        if model is not None:
            model.to('cpu')
        del model
        empty_cuda_cache()

## Select the shared random test sample

This cell samples exactly `SAMPLES_PER_DATASET` usable rows from every configured `test.tsv`. Run it once; all model cells reuse `selected_examples` without resampling.

In [ ]:
selected_examples, selected_dataset_names = sample_test_rows(
    TEST_DATASETS, SAMPLES_PER_DATASET, RANDOM_SEED
)
sample_frame = pd.DataFrame([
    {
        'dataset': dataset, 'file': example.audio_path.name,
        'duration_seconds': round(sf.info(str(example.audio_path)).duration, 2),
        'reference': example.transcript, 'path': str(example.audio_path),
    }
    for dataset, example in zip(selected_dataset_names, selected_examples, strict=True)
])
display(
    sample_frame.style.set_properties(
        subset=['reference'],
        **{'white-space': 'pre-wrap', 'text-align': 'right', 'min-width': '24rem'},
    )
)
if SHOW_AUDIO_PLAYERS:
    for row, example in zip(sample_frame.to_dict('records'), selected_examples, strict=True):
        display(Markdown(f"**{row['dataset']} — `{row['file']}`**  \n{row['reference']}"))
        display(Audio(filename=str(example.audio_path)))

## 1. Normal Whisper-small

In [ ]:
normal_whisper_results = run_whisper(
    '1. Normal Whisper-small', WHISPER_NORMAL_CHECKPOINT, WHISPER_NORMAL_PROCESSOR
)

## 2. Multiconditioned Whisper-small

In [ ]:
multiconditioned_whisper_results = run_whisper(
    '2. Multiconditioned Whisper-small',
    WHISPER_MULTICONDITIONED_CHECKPOINT,
    WHISPER_MULTICONDITIONED_PROCESSOR,
)

## 3. Dual-view fusion model

In [ ]:
fusion_results = run_fusion()

## 4. Whisper-medium

In [ ]:
whisper_medium_results = run_whisper(
    '4. Whisper-medium', WHISPER_MEDIUM_CHECKPOINT, WHISPER_MEDIUM_PROCESSOR
)

## 5. Normal FastConformer

In [ ]:
normal_fastconformer_results = run_fastconformer(
    '5. Normal FastConformer', FASTCONFORMER_NORMAL_CHECKPOINT
)

## 6. Multiconditioned FastConformer

In [ ]:
multiconditioned_fastconformer_results = run_fastconformer(
    '6. Multiconditioned FastConformer', FASTCONFORMER_MULTICONDITIONED_CHECKPOINT
)

## 7. Multiconditioned + IranSeda Whisper-small

In [ ]:
multiconditioned_iranseda_whisper_results = run_whisper(
    '7. Multiconditioned + IranSeda Whisper-small',
    WHISPER_MULTICONDITIONED_IRANSEDA_CHECKPOINT,
    WHISPER_MULTICONDITIONED_IRANSEDA_PROCESSOR,
)

## Comparison
This table uses only the shared random sample, so it is a demo comparison rather than a replacement for full-test evaluation.

In [ ]:
expected_models = [
    '1. Normal Whisper-small', '2. Multiconditioned Whisper-small', '3. Fusion model',
    '4. Whisper-medium', '5. Normal FastConformer', '6. Multiconditioned FastConformer',
    '7. Multiconditioned + IranSeda Whisper-small',
]
missing = [name for name in expected_models if name not in RESULTS]
if missing:
    raise RuntimeError(f'Run the missing model cells first: {missing}')
comparison = pd.DataFrame([
    {
        'model': name, 'WER': RESULTS[name]['wer'], 'CER': RESULTS[name]['cer'],
        'seconds': RESULTS[name]['elapsed_seconds'],
        'seconds/audio': RESULTS[name]['elapsed_seconds'] / len(selected_examples),
    }
    for name in expected_models
]).sort_values('WER', ignore_index=True)
display(comparison.style.format({'WER': '{:.4f}', 'CER': '{:.4f}', 'seconds': '{:.2f}', 'seconds/audio': '{:.3f}'}))
empty_cuda_cache()